In [ ]:
!pip install nltk gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.7 MB/s eta 0:00:00


In [ ]:
# Run this cell if the required packages are not installed.
# !pip install pandas numpy gensim nltk torch transformers scikit-learn

import re
import os
import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
tweets = [
    "I love this phone, the battery life is amazing!",
    "The phone battery died after two hours. Very bad.",
    "Apple launched a new iPhone today.",
    "I ate an apple after lunch.",
    "The movie was amazing and the acting was excellent.",
    "This movie is terrible, I want my money back.",
    "Python is great for machine learning.",
    "I am learning Python programming for my project.",
    "The bank approved my loan yesterday.",
    "I went to the river bank to take pictures.",
    "AI is changing the future of software engineering.",
    "The software update fixed many security bugs.",
    "Cybersecurity is important for every company.",
    "Hackers attacked the company's server last night.",
    "This laptop is fast but the keyboard is uncomfortable.",
    "The new model understands the tweet context very well."
]

df = pd.DataFrame({'tweet': tweets})
df.head(20)


,tweet
0,"I love this phone, the battery life is amazing!"
1,The phone battery died after two hours. Very bad.
2,Apple launched a new iPhone today.
3,I ate an apple after lunch.
4,The movie was amazing and the acting was excel...
5,"This movie is terrible, I want my money back."
6,Python is great for machine learning.
7,I am learning Python programming for my project.
8,The bank approved my loan yesterday.
9,I went to the river bank to take pictures.


In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_tweet(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\\w+', '', text)

    # Keep hashtag text but remove '#'
    text = re.sub(r'#', '', text)

    # Word tokenization using NLTK
    tokens = word_tokenize(text)

    # Keep alphabetic words and remove stop words
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return tokens

df['tokens'] = df['tweet'].apply(preprocess_tweet)

df[['tweet', 'tokens']]


,tweet,tokens
0,"I love this phone, the battery life is amazing!","[love, phone, battery, life, amazing]"
1,The phone battery died after two hours. Very bad.,"[phone, battery, died, two, hours, bad]"
2,Apple launched a new iPhone today.,"[apple, launched, new, iphone, today]"
3,I ate an apple after lunch.,"[ate, apple, lunch]"
4,The movie was amazing and the acting was excel...,"[movie, amazing, acting, excellent]"
5,"This movie is terrible, I want my money back.","[movie, terrible, want, money, back]"
6,Python is great for machine learning.,"[python, great, machine, learning]"
7,I am learning Python programming for my project.,"[learning, python, programming, project]"
8,The bank approved my loan yesterday.,"[bank, approved, loan, yesterday]"
9,I went to the river bank to take pictures.,"[went, river, bank, take, pictures]"


In [ ]:
for original, tokens in zip(df['tweet'], df['tokens']):
    print('Original :', original)
    print('Cleaned  :', tokens)
    print('-' * 70)


Original : I love this phone, the battery life is amazing!
Cleaned  : ['love', 'phone', 'battery', 'life', 'amazing']
----------------------------------------------------------------------
Original : The phone battery died after two hours. Very bad.
Cleaned  : ['phone', 'battery', 'died', 'two', 'hours', 'bad']
----------------------------------------------------------------------
Original : Apple launched a new iPhone today.
Cleaned  : ['apple', 'launched', 'new', 'iphone', 'today']
----------------------------------------------------------------------
Original : I ate an apple after lunch.
Cleaned  : ['ate', 'apple', 'lunch']
----------------------------------------------------------------------
Original : The movie was amazing and the acting was excellent.
Cleaned  : ['movie', 'amazing', 'acting', 'excellent']
----------------------------------------------------------------------
Original : This movie is terrible, I want my money back.
Cleaned  : ['movie', 'terrible', 'want', 'money

In [ ]:
tokenized_tweets = df['tokens'].tolist()

print('Number of tweets:', len(tokenized_tweets))
print('First cleaned tweet:', tokenized_tweets[0])


Number of tweets: 16
First cleaned tweet: ['love', 'phone', 'battery', 'life', 'amazing']


In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

word = 'bank'
vector = w2v_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

print('\nSimilar words:')
print(w2v_model.wv.most_similar(word, topn=5))


Word: bank
Vector shape: (100,)
First 10 values: [ 0.00038309  0.0032622  -0.00728072 -0.00151712  0.00731176  0.00553623
 -0.00336169  0.00454942 -0.00868977  0.00471174]

Similar words:
[('yesterday', 0.24527476727962494), ('well', 0.23155668377876282), ('amazing', 0.22988075017929077), ('excellent', 0.22535552084445953), ('fast', 0.20152848958969116)]


In [ ]:
def load_glove(glove_path):
    embeddings = {}

    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.rstrip().split(' ')
            word = values[0]
            vector = np.asarray(values[1:], dtype=np.float32)
            embeddings[word] = vector

    return embeddings

glove_path = 'glove.6B.100d.txt'

if os.path.exists(glove_path):
    glove = load_glove(glove_path)
    word = 'bank'
    print('Word:', word)
    print('Vector shape:', glove[word].shape)
    print('First 10 values:', glove[word][:10])
else:
    print('GloVe file not found.')
    print('Place glove.6B.100d.txt in the notebook directory to run this section.')


GloVe file not found.
Place glove.6B.100d.txt in the notebook directory to run this section.


In [ ]:
from gensim.models import FastText

fasttext_model = FastText(
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

fasttext_model.build_vocab(corpus_iterable=tokenized_tweets)
fasttext_model.train(
    corpus_iterable=tokenized_tweets,
    total_examples=len(tokenized_tweets),
    epochs=100
)

word = 'bank'
vector = fasttext_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

unknown_word = 'amazzzzing'
unknown_vector = fasttext_model.wv[unknown_word]
print('\nFastText can generate a vector for:', unknown_word)
print('Vector shape:', unknown_vector.shape)


Word: bank
Vector shape: (100,)
First 10 values: [-3.7789496e-03 -2.8636651e-03 -2.1460950e-03 -1.8785144e-03
  1.7680831e-03 -7.3039155e-05  1.1883718e-03 -1.8888668e-03
  9.4088893e-05 -2.4465250e-03]

FastText can generate a vector for: amazzzzing
Vector shape: (100,)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

contextual_tweets = [
    'I deposited money in the bank.',
    'I sat near the river bank.'
]

encoded = bert_tokenizer(
    contextual_tweets,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    outputs = bert_model(**encoded)

token_embeddings = outputs.last_hidden_state

print('Token embedding shape:', token_embeddings.shape)
print('BERT hidden size:', token_embeddings.shape[-1])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token embedding shape: torch.Size([2, 9, 768])
BERT hidden size: 768


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

bert_tweet_embeddings = mean_pool(
    token_embeddings,
    encoded['attention_mask']
)

print('Tweet embedding shape:', bert_tweet_embeddings.shape)
print('First tweet embedding, first 10 values:')
print(bert_tweet_embeddings[0][:10].numpy())


Tweet embedding shape: torch.Size([2, 768])
First tweet embedding, first 10 values:
[ 0.4423436  -0.15134208 -0.13635194  0.14299281  0.2534433  -0.04802973
 -0.19784266  0.6531207   0.23939732 -0.55328774]


In [ ]:
from transformers import AutoTokenizer, AutoModel

gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gpt_model = AutoModel.from_pretrained('gpt2')
gpt_model.eval()

gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_encoded = gpt_tokenizer(
    contextual_tweets,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

with torch.no_grad():
    gpt_outputs = gpt_model(**gpt_encoded)

gpt_token_embeddings = gpt_outputs.last_hidden_state

gpt_tweet_embeddings = mean_pool(
    gpt_token_embeddings,
    gpt_encoded['attention_mask']
)

print('GPT-2 token embedding shape:', gpt_token_embeddings.shape)
print('GPT-2 tweet embedding shape:', gpt_tweet_embeddings.shape)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 token embedding shape: torch.Size([2, 7, 768])
GPT-2 tweet embedding shape: torch.Size([2, 768])


In [ ]:
np.save('bert_tweet_embeddings.npy', bert_tweet_embeddings.numpy())
np.save('gpt2_tweet_embeddings.npy', gpt_tweet_embeddings.numpy())

print('Saved: bert_tweet_embeddings.npy')
print('Saved: gpt2_tweet_embeddings.npy')


Saved: bert_tweet_embeddings.npy
Saved: gpt2_tweet_embeddings.npy
